In [10]:
# ── 1. Repos & deps ──────────────────────────────────────────────────────────
!git clone https://github.com/goombalab/hnet.git
!git clone --recurse-submodules https://github.com/Chaoqi-LIU/oat.git
!rm -rf congenial-adventure && git clone https://github.com/SDogya/congenial-adventure.git && cp -r congenial-adventure/. . && rm -rf congenial-adventure

import os
from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('wandb')

!uv add "zarr<3.0.0" dill einops numba vector-quantize-pytorch accelerate huggingface_hub "robomimic<0.3.0" torchvision wrapt pillow pandas diffusers
!uv sync

# Patch OAT's lr_scheduler.py: newer diffusers removed Union/Optional/Optimizer exports
p = 'oat/oat/model/common/lr_scheduler.py'
txt = open(p).read()
marker = 'from diffusers.optimization import ('
if marker in txt and 'from typing import Union' not in txt:
    idx = txt.index(marker)
    end_idx = txt.index(')', idx) + 1
    header = (
        'from typing import Union, Optional\n'
        'from torch.optim import Optimizer\n'
        'from diffusers.optimization import SchedulerType, TYPE_TO_SCHEDULER_FUNCTION'
    )
    open(p, 'w').write(txt[:idx] + header + txt[end_idx:])
    print('lr_scheduler.py patched OK')
else:
    print('lr_scheduler.py already clean, skipping')

# Patch OAT: заглушить print'ы про нормализацию
for path in [
    'oat/oat/perception/state_encoder.py',
    'oat/oat/perception/robomimic_vision_encoder.py',
]:
    txt = open(path).read()
    txt = txt.replace(
        'print(warning_msg(f"no normalizer params for port {port}, skipping normalization."))',
        'pass  # suppressed'
    ).replace(
        'print(f"no normalizer params for port {port}, skipping normalization.")',
        'pass  # suppressed'
    )
    open(path, 'w').write(txt)
print('normalizer prints suppressed OK')


fatal: destination path 'hnet' already exists and is not an empty directory.
fatal: destination path 'oat' already exists and is not an empty directory.
Cloning into 'congenial-adventure'...
remote: Enumerating objects: 434, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 434 (delta 0), reused 1 (delta 0), pack-reused 431 (from 2)
Receiving objects: 100% (434/434), 81.58 MiB | 43.00 MiB/s, done.
Resolving deltas: 100% (190/190), done.
Resolved 118 packages in 100ms                                       
Audited 115 packages in 2ms                                          
Resolved 118 packages in 0.94ms
Audited 115 packages in 2ms
lr_scheduler.py already clean, skipping
normalizer prints suppressed OK


In [2]:
!ls -ld /kaggle/working/oat/data/libero/*

drwxr-xr-x 4 root root       4096 Apr 13 01:28 /kaggle/working/oat/data/libero/libero10_N500.zarr
-rw-r--r-- 1 root root 3528193175 Apr 13 01:28 /kaggle/working/oat/data/libero/libero10_N500.zarr.zip
drwxr-xr-x 3 root root       4096 Apr 13 01:27 /kaggle/working/oat/data/libero/__MACOSX
-rw-r--r-- 1 root root         24 Apr 13 01:27 /kaggle/working/oat/data/libero/README.md


In [2]:
# ── 2. Dataset (скачиваем прямо туда, куда смотрит OAT по дефолту) ───────────
import os
from huggingface_hub import snapshot_download
from huggingface_hub import login
hf_token = UserSecretsClient().get_secret('hugface')

if hf_token:
    login(token=hf_token)
else:
    print("Ошибка: Секрет 'hugface' не найден.")


os.makedirs('/kaggle/working/oat/data/libero', exist_ok=True)
snapshot_download(
    repo_id='chaoqi-liu/libero10_N500.zarr',
    repo_type='dataset',
    local_dir='/kaggle/working/oat/data/libero'
)



# Распаковываем архив прямо в ту же папку
!unzip -o -q /kaggle/working/oat/data/libero/libero10_N500.zarr.zip -d /kaggle/working/oat/data/libero/

# Проверяем, что папка появилась
!ls -ld /kaggle/working/oat/data/libero/*

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

replace /kaggle/working/oat/data/libero/libero10_N500.zarr/.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C
drwxr-xr-x 4 root root       4096 Apr 12 21:50 /kaggle/working/oat/data/libero/libero10_N500.zarr
-rw-r--r-- 1 root root 3528193175 Apr 12 21:50 /kaggle/working/oat/data/libero/libero10_N500.zarr.zip
drwxr-xr-x 3 root root       4096 Apr 12 21:49 /kaggle/working/oat/data/libero/__MACOSX
-rw-r--r-- 1 root root         24 Apr 12 21:58 /kaggle/working/oat/data/libero/README.md


In [2]:
# ── 3. Train OAT tokenizer (~2-3h, 300 epochs) ───────────────────────────────
!uv run python oat/scripts/run_workspace.py \
    --config-name=train_oattok \
    task/tokenizer=libero/libero10 \
    training.num_epochs=300 \
    logging.project=VLA-experiment \
    task.tokenizer.dataset.zarr_path="/kaggle/working/oat/data/libero/libero10_N500.zarr"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: sebersehmer (sebersehmer-nopeinc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/oat/output/20260412/225721_train_oattok_libero10_N500/wandb/run-20260412_225733-ladyzfee
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run 20260412.225721_train_oattok_libero10_N500
wandb: ⭐️ View project at https://wandb.ai/sebersehmer-nopeinc/VLA-experiment
wandb: 🚀 View run at https://wandb.ai/sebersehmer-nopeinc/VLA-experiment/runs/ladyzfee
wandb: ⢿ updating run metadata (0.0s)               
wandb: ⣻ updating run metadata (0.0s)
wandb: ⣽ updating run metadata (0.0s)
wandb: ⣾ updating run metadata (0.0s)
wandb: ⣷ uploading history steps 919-967, summary, console li

In [ ]:

# ── 4. Train FD-DRAT ─────────────────────────────────────────────────────────
import glob
ckpt_candidates = sorted(
    glob.glob('/kaggle/working/output/**/model.ckpt', recursive=True) +
    glob.glob('/kaggle/working/**/model.ckpt', recursive=True)
)
TOK = ckpt_candidates[-1] if ckpt_candidates else ''
print(f'Using tokenizer: {TOK}')

!HYDRA_FULL_ERROR=1 MPLBACKEND=agg uv run run.py \
    strategy=single_gpu \
    model.tokenizer_ckpt={TOK} \
    dataset_path=/kaggle/working/oat/data/libero/libero10_N500.zarr \
    batch_size=16


In [6]:
!find /kaggle/working/output /kaggle/working -name 'model.ckpt' 2>/dev/null | sort | tail -1

/kaggle/working/src/model/model.ckpt
